# 🌿 BioCiclo IA - Treinamento YOLO no Google Colab (GPU Grátis)
**Disciplina:** Fundamentos da Inteligência Artificial - UFS  
**Contexto:** BioCiclo Urbano (Aracaju / EMSURB)  

> **Instruções:**  
> 1. No menu superior do Colab, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** > Selecione **T4 GPU**.
> 2. Execute as células abaixo em sequência (Shift + Enter).

### 1. Verificar GPU e Instalar Dependências

In [ ]:
!nvidia-smi
!pip install ultralytics roboflow

### 2. Baixar o Dataset Garbage Classification 3 v2 (GC1)
Insira sua chave gratuita do Roboflow abaixo:

In [ ]:
from roboflow import Roboflow

# Substitua pela sua chave do Roboflow (obtenha em app.roboflow.com/settings/api)
ROBOFLOW_API_KEY = "MCFCWzlXnoQWlcDxiV2W"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("material-identification").project("garbage-classification-3")
version = project.version(2)
dataset = version.download("yolov8")
print("Dataset baixado com sucesso em:", dataset.location)

### 3. Treinar o Modelo YOLO (Fine-Tuning com Transfer Learning)
Utilizaremos o `yolo11n.pt` pré-treinado adaptado para as classes do resíduo.

In [ ]:
from ultralytics import YOLO

# Carregar modelo pré-treinado
model = YOLO('yolo11n.pt')

# Treinamento acelerado por GPU
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project="biociclo_runs",
    name="treino_yolo_biociclo"
)

### 4. Visualizar Matriz de Confusão e Baixar `best.pt`

In [ ]:
import glob
from google.colab import files
from IPython.display import Image, display

# Localizar automaticamente os arquivos gerados pelo YOLO
best_pts = glob.glob('/content/**/weights/best.pt', recursive=True)
matrices = glob.glob('/content/**/confusion_matrix.png', recursive=True)
results_imgs = glob.glob('/content/**/results.png', recursive=True)

print("Arquivos encontrados no Colab:")
print("- Pesos (best.pt):", best_pts)
print("- Matriz de Confusao:", matrices)

# Exibir Matriz de Confusao
if matrices:
    display(Image(filename=matrices[0]))
if results_imgs:
    display(Image(filename=results_imgs[0]))

# Baixar o melhor modelo treinado para o computador
if best_pts:
    print(f"Iniciando download de {best_pts[0]}...")
    files.download(best_pts[0])
else:
    print("Arquivo best.pt nao encontrado. Verifique se a Celula 3 terminou com sucesso.")